# Qwen2.5-7B-Instruct: Yoruba → English Idiom Translation
Machine Translation of Yorùbá Idiomatic Expressions  
**Model:** Qwen2.5-7B-Instruct

**Settings:** Zero-Shot and Few-Shot (20 in-context examples)  
**Metrics:** BLEU · chrF+ · BERTScore F1  



In [1]:
!pip install transformers torch accelerate pandas tqdm sacrebleu bert-score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 8.4 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# Configuration

MODEL_NAME    = "Qwen/Qwen2.5-7B-Instruct"
DATASET_PATH  = "/content/drive/MyDrive/Yoruba Idioms.csv"
OUTPUT_DIR    = "/content/drive/MyDrive/results"
FEW_SHOT_K    = 20
RANDOM_SEED   = 42
MAX_TOKENS    = 50
TEMPERATURE   = 0.2

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete")

Setup complete


In [3]:
print("Loading Qwen2.5-7B-Instruct ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Done {model.device}")

Loading Qwen2.5-7B-Instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Done cuda:0


In [4]:
def load_dataset(path: str) -> pd.DataFrame:
    """Load and clean the Yorùbá idioms dataset."""
    df = pd.read_csv(path)
    df.columns = ['idiom', 'literal_meaning', 'idiomatic_meaning']
    df = df.dropna(subset=['idiom', 'idiomatic_meaning']).reset_index(drop=True)
    return df

df = load_dataset(DATASET_PATH)
print(f"Dataset loaded: {len(df)} samples")
print(df.head())

Dataset loaded: 104 samples
                   idiom                                 literal_meaning  \
0      Ta téru nípàá                          Kick the calico fabric   
1           Júbà ehoro                          Pay homage to the hare   
2         Fi àáké kọ́rí                         Hang an axe on the head   
3  Tutọ́ sókè fojú gbà á  Spit upward and let it land back on your face,   
4            Ru igi Oyin        Carry a tree with honey bees on the head   

             idiomatic_meaning  
0                       To die  
1                  To run away  
2            Insist completely  
3        Be excessively angry.  
4  Throw oneself into trouble.  


In [5]:
# Prompt Design

SYSTEM_PROMPT = (
 "You are a Yoruba linguist and cultural translator. "
"Given a Yoruba idiom, provide its English idiomatic meaning—"
"the figurative or intended sense, not a literal word-for-word translation. "
"Output only the meaning in one short sentence with no explanation."
)

USER_PROMPT_TEMPLATE = "Yorùbá Idiom: {idiom}\nEnglish Meaning:"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [7]:
def generate_response(messages: list) -> str:
    """Generate output from Qwen given chat messages."""

    # safety check (VERY important)
    if tokenizer is None or model is None:
        raise ValueError("Model or tokenizer not initialized. Run loading cell first.")

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [8]:
def translate_zero_shot(idiom: str) -> str:
    """Direct machine translation using zero-shot prompting."""
    try:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": USER_PROMPT_TEMPLATE.format(idiom=idiom)}
        ]
        return generate_response(messages)
    except Exception as e:
        print(f"[Zero-Shot Error] {idiom[:30]}... → {e}")
        return ""


def translate_few_shot(idiom: str, examples: pd.DataFrame) -> str:
    """Few-shot prompting with in-context examples."""
    try:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

        for _, row in examples.iterrows():
            messages.append({
                "role": "user",
                "content": USER_PROMPT_TEMPLATE.format(idiom=row['idiom'])
            })
            messages.append({
                "role": "assistant",
                "content": row['idiomatic_meaning']
            })

        messages.append({
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(idiom=idiom)
        })

        return generate_response(messages)
    except Exception as e:
        print(f"[Few-Shot Error] {idiom[:30]}... → {e}")
        return ""

In [9]:
assert tokenizer is not None, "Tokenizer not initialized"
assert model is not None, "Model not initialized"

In [10]:
# Split dataset into few-shot samples and test set
few_shot_examples = df.sample(FEW_SHOT_K, random_state=RANDOM_SEED)
test_df           = df.drop(few_shot_examples.index).reset_index(drop=True)

print(f"Few-shot examples : {len(few_shot_examples)}")
print(f"Test samples      : {len(test_df)}\n")

zero_shot_predictions = []
few_shot_predictions  = []

for idiom in tqdm(test_df['idiom'], desc="Translating"):
    zero_shot_predictions.append(translate_zero_shot(idiom))
    few_shot_predictions.append(translate_few_shot(idiom, few_shot_examples))

# Save predictions
test_df['zero_shot_prediction'] = zero_shot_predictions
test_df['few_shot_prediction']  = few_shot_predictions

predictions_path = os.path.join(OUTPUT_DIR, "qwen25_predictions.csv")
test_df.to_csv(predictions_path, index=False)
print(f"\nPredictions saved to {predictions_path}")

Few-shot examples : 20
Test samples      : 84



Translating: 100%|██████████| 84/84 [00:53<00:00,  1.56it/s]



Predictions saved to /content/drive/MyDrive/results/qwen25_predictions.csv


In [11]:
from sacrebleu.metrics import BLEU, CHRF
from bert_score import score as bert_score_fn

def evaluate(predictions: list, references: list, label: str) -> dict:
    """Evaluate predictions using BLEU, chrF+, and BERTScore F1."""
    bleu      = BLEU()
    chrf_plus = CHRF(word_order=1)   # chrF+

    bleu_score      = bleu.corpus_score(predictions, [references]).score
    chrf_plus_score = chrf_plus.corpus_score(predictions, [references]).score
    _, _, F1        = bert_score_fn(predictions, references, lang="en", verbose=False)

    return {
        "setting"      : label,
        "BLEU"         : round(bleu_score, 4),
        "chrF+"        : round(chrf_plus_score, 4),
        "BERTScore_F1" : round(F1.mean().item(), 4)
    }


# Evaluate both settings
references   = test_df['idiomatic_meaning'].tolist()
zero_results = evaluate(test_df['zero_shot_prediction'].tolist(), references, "Zero-Shot")
few_results  = evaluate(test_df['few_shot_prediction'].tolist(),  references, "Few-Shot")

results_df = pd.DataFrame([zero_results, few_results])

# Save metrics
metrics_path = os.path.join(OUTPUT_DIR, "qwen25_metrics.csv")
results_df.to_csv(metrics_path, index=False)

# Display results
print("\n" + "=" * 60)
print("      Qwen2.5-7B EVALUATION RESULTS — YO → EN")
print("=" * 60)
print(results_df.to_string(index=False))
print("=" * 60)
print(f"\nMetrics saved to {metrics_path}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



      Qwen2.5-7B EVALUATION RESULTS — YO → EN
  setting   BLEU   chrF+  BERTScore_F1
Zero-Shot 0.3386 11.8340        0.8673
 Few-Shot 0.5723 11.4997        0.8697

Metrics saved to /content/drive/MyDrive/results/qwen25_metrics.csv


In [12]:
predictions_path = os.path.join(OUTPUT_DIR, "qwen2.5_predictions.csv")
test_df.to_csv(predictions_path, index=False)

print(f"Predictions saved to {predictions_path}")

Predictions saved to /content/drive/MyDrive/results/qwen2.5_predictions.csv


In [13]:
from google.colab import files

files.download(predictions_path)
files.download(metrics_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/results/qwen2.5_predictions.csv")
df.tail(20)

,idiom,literal_meaning,idiomatic_meaning,zero_shot_prediction,few_shot_prediction
64,Fi ara sílẹ̀,Put body down,"to be caught unawares, unalert, not cautious",To act as if nothing happened or to pretend no...,to hide one's face
65,Gbé ara sọ,put body up,to stretch oneself beyond assumed capability,To cover up the mistake by making another one.,To hide something
66,Fi ara wé,Use body to compare,to imitate or copy,It's time for you to learn.,To present oneself to someone
67,Àṣírí tú,secret open,a secret is exposed,It's too late to change things now.,It's too late
68,Tẹ orí gba aṣọ,press head to accept cloth,to die,He is in a good mood or feeling happy.,To change one's mind
69,Pa àtẹ́,display tray,to display a behaviour in a derogatory way,To play along or go along with something.,to be late
70,Awọ ò kájú ìlù,leather does not cover drum,Lack or inability to make ends meet,A meeting of minds or agreement is reached.,A meeting to discuss a matter
71,Dé ara mọ́ awo olóókan,Sealing oneself within the secret of the one-h...,to know one’s limit,He is like a doctor who diagnoses himself.,To take the blame for someone else's mistake
72,Ní àyà,have chest,to be brave and courageous,To be on guard or to be cautious.,in trouble
73,Fi àyà tìí,use chest to push it,to endure/ stand with,To bite off more than you can chew.,To relieve oneself (go to the bathroom)
